[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/40_mlops/01_serving_with_fastapi.ipynb)

# Bài 01 — Phục vụ mô hình bằng FastAPI

**🎯 Mục tiêu**
- Thiết kế **hợp đồng API** cho mô hình: endpoint, schema đầu vào/đầu ra, mã lỗi, phiên bản.
- Viết dịch vụ FastAPI hoàn chỉnh: nạp mô hình **một lần** khi khởi động (kiểm tra hash), kiểm tra đầu vào bằng pydantic, dự đoán đơn lẻ và theo lô, **ghi log dự đoán**.
- Kiểm thử bằng `TestClient` (kể cả kiểm thử **parity** với mô hình gốc), chạy server thật bằng uvicorn và đo độ trễ qua HTTP.
- Hiểu các lựa chọn hiệu năng: theo lô vs đơn lẻ, `def` vs `async def`, nạp mô hình mỗi request (anti-pattern).

Cần `pip install -e ".[mlops]"` (FastAPI, uvicorn, httpx, pydantic).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/40_mlops

In [ ]:
import hashlib
import importlib.util
import json
import socket
import subprocess
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_course.datasets import make_customer_churn

warnings.filterwarnings("ignore", message="Using `httpx` with")       # cảnh báo đổi tên gói của starlette, vô hại
try:
    import httpx
    from fastapi.testclient import TestClient
except ImportError as e:
    raise ImportError('Bài này cần FastAPI + httpx: pip install -e ".[mlops]"') from e

SERVICE_DIR = Path("artifacts/churn_service")                          # thư mục này đã nằm trong .gitignore
(SERVICE_DIR / "model").mkdir(parents=True, exist_ok=True)

## 1. Hợp đồng API

Trước khi viết code, thống nhất **hợp đồng** với bên gọi (app, backend, đội dữ liệu):

| Endpoint | Phương thức | Đầu vào | Đầu ra | Mã lỗi |
|---|---|---|---|---|
| `/health` | GET | — | `{"status": "ok", "model_loaded": true}` | — |
| `/v1/model` | GET | — | tên, phiên bản, đặc trưng, metric của mô hình | — |
| `/v1/predict` | POST | JSON một khách hàng (9 trường, có ràng buộc) | `request_id`, `churn_probability`, `model_version` | 422 nếu đầu vào sai |
| `/v1/predict/batch` | POST | `{"customers": [...]}` (1 → 1000 khách) | danh sách dự đoán | 422 |

Nguyên tắc:
- **Phiên bản trong URL** (`/v1/`): đổi schema đầu vào → `/v2/`, giữ `/v1/` cho tới khi mọi bên gọi chuyển xong.
- Trả về **`model_version`** và **`request_id`** — để truy vết một dự đoán về đúng mô hình, và nối với nhãn thật khi có (bài 03).
- Từ chối đầu vào sai bằng **422** kèm mô tả lỗi; không bao giờ trả stack trace ra ngoài.

## 2. Chuẩn bị gói mô hình

Như chương 39: pipeline đầy đủ + `metadata.json` (phiên bản, schema, metric, hash file).

In [ ]:
FEATURES = ["tenure_months", "contract", "internet_service", "monthly_charges", "total_charges",
            "payment_method", "support_calls", "senior", "n_products"]
NUMERIC = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
CATEGORICAL = ["contract", "internet_service", "payment_method", "senior"]

data = make_customer_churn(8000, seed=0)
X_tr, X_te, y_tr, y_te = train_test_split(data[FEATURES], data.churn, test_size=0.25, random_state=0, stratify=data.churn)
model = make_pipeline(ColumnTransformer([("num", StandardScaler(), NUMERIC),
                                         ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
                      LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)

model_path = SERVICE_DIR / "model" / "model.joblib"
joblib.dump(model, model_path)
metadata = {"name": "churn", "version": "1.0.0", "features": FEATURES,
            "metrics": {"test_roc_auc": round(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]), 4)},
            "sklearn_version": sklearn.__version__, "model_sha256": hashlib.sha256(model_path.read_bytes()).hexdigest(),
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
(SERVICE_DIR / "model" / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(json.dumps({k: metadata[k] for k in ("name", "version", "metrics")}), "→", sorted(p.name for p in (SERVICE_DIR / "model").iterdir()))

## 3. Viết dịch vụ

Dịch vụ là một **file Python riêng** (không phải code trong notebook) — đúng như khi triển khai. Ô dưới ghi file `artifacts/churn_service/app.py`; hãy mở file đó để đọc lại.

Các quyết định thiết kế:
- **Nạp mô hình một lần** trong `lifespan` (khi server khởi động), kiểm tra hash; nạp thất bại → server **không** khởi động (tốt hơn chạy với mô hình hỏng).
- Schema pydantic giống chương 39; `extra="forbid"` bắt lỗi gõ sai tên trường; lô giới hạn 1000 dòng (chống request quá lớn).
- Endpoint dùng **`def`** (không phải `async def`): `predict_proba` là tác vụ CPU chặn; FastAPI chạy hàm `def` trong threadpool, còn `async def` chứa code chặn sẽ **khoá event loop**, làm mọi request khác chờ.
- Middleware gắn thời gian xử lý vào header; mỗi dự đoán ghi một dòng JSON vào **log dự đoán** — dữ liệu cho giám sát (bài 03).

In [ ]:
%%writefile artifacts/churn_service/app.py
"""Dịch vụ dự đoán khách hàng rời bỏ (churn) — FastAPI."""
import hashlib
import json
import os
import threading
import time
import uuid
from contextlib import asynccontextmanager
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

import joblib
import pandas as pd
from fastapi import FastAPI, Request
from pydantic import BaseModel, ConfigDict, Field

HERE = Path(__file__).parent
MODEL_DIR = Path(os.environ.get("MODEL_DIR", HERE / "model"))
PREDICTION_LOG = Path(os.environ.get("PREDICTION_LOG", HERE / "predictions.jsonl"))
STATE: dict = {}
LOG_LOCK = threading.Lock()


class CustomerFeatures(BaseModel):
    model_config = ConfigDict(extra="forbid")
    tenure_months: int = Field(ge=0, le=600)
    contract: Literal["month-to-month", "one_year", "two_year"]
    internet_service: Literal["dsl", "fiber", "none"]
    monthly_charges: float = Field(gt=0, le=1000)
    total_charges: float = Field(ge=0)
    payment_method: Literal["bank_transfer", "cash", "credit_card", "e-wallet"]
    support_calls: int = Field(ge=0, le=100)
    senior: bool
    n_products: int = Field(ge=1, le=20)


class BatchRequest(BaseModel):
    customers: list[CustomerFeatures] = Field(min_length=1, max_length=1000)


class Prediction(BaseModel):
    request_id: str
    churn_probability: float
    model_version: str


def load_model():
    meta = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))
    if hashlib.sha256((MODEL_DIR / "model.joblib").read_bytes()).hexdigest() != meta["model_sha256"]:
        raise RuntimeError("model.joblib không khớp hash trong metadata — từ chối khởi động")
    return joblib.load(MODEL_DIR / "model.joblib"), meta


@asynccontextmanager
async def lifespan(app: FastAPI):
    STATE["model"], STATE["meta"] = load_model()                       # nạp MỘT lần khi khởi động
    yield
    STATE.clear()


app = FastAPI(title="Churn prediction API", version="1.0.0", lifespan=lifespan)


@app.middleware("http")
async def timing_header(request: Request, call_next):
    t0 = time.perf_counter()
    response = await call_next(request)
    response.headers["X-Process-Time-ms"] = f"{(time.perf_counter() - t0) * 1000:.2f}"
    return response


def predict_rows(rows: list[dict]) -> list[Prediction]:
    t0 = time.perf_counter()
    probs = STATE["model"].predict_proba(pd.DataFrame(rows))[:, 1].tolist()
    latency_ms = (time.perf_counter() - t0) * 1000
    version = STATE["meta"]["version"]
    ids = [uuid.uuid4().hex for _ in rows]
    now = datetime.now(timezone.utc).isoformat(timespec="milliseconds")
    lines = [json.dumps({"ts": now, "request_id": i, "model_version": version, "features": r,
                         "churn_probability": p, "latency_ms": latency_ms}) for i, r, p in zip(ids, rows, probs)]
    with LOG_LOCK, PREDICTION_LOG.open("a", encoding="utf-8") as f:   # nhiều thread cùng ghi → khoá
        f.write("\n".join(lines) + "\n")
    return [Prediction(request_id=i, churn_probability=p, model_version=version) for i, p in zip(ids, probs)]


@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": "model" in STATE}


@app.get("/v1/model")
def model_info():
    return {k: STATE["meta"][k] for k in ("name", "version", "features", "metrics", "created_utc")}


@app.post("/v1/predict", response_model=Prediction)
def predict(customer: CustomerFeatures):
    return predict_rows([customer.model_dump()])[0]


@app.post("/v1/predict/batch", response_model=list[Prediction])
def predict_batch(batch: BatchRequest):
    return predict_rows([c.model_dump() for c in batch.customers])

## 4. Kiểm thử bằng TestClient

`TestClient` gọi ứng dụng **trong cùng tiến trình** — không cần mở cổng mạng — nên chạy nhanh, dùng được trong pytest/CI (bài 02). Dùng `with TestClient(app)` để `lifespan` (nạp mô hình) được chạy.

In [ ]:
def load_service(path):
    spec = importlib.util.spec_from_file_location("churn_service_app", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


service = load_service(SERVICE_DIR / "app.py")
service.PREDICTION_LOG.unlink(missing_ok=True)                         # bắt đầu với log trống
records = json.loads(X_te.iloc[:200].to_json(orient="records"))        # JSON thuần (int/float/bool Python), như client gửi

with TestClient(service.app) as client:
    print("GET /health      →", client.get("/health").json())
    print("GET /v1/model    →", client.get("/v1/model").json()["metrics"])
    r = client.post("/v1/predict", json=records[0])
    print("POST /v1/predict →", r.status_code, r.json(), "| header thời gian:", r.headers["X-Process-Time-ms"], "ms")
    bad = client.post("/v1/predict", json={**records[0], "contract": "monthly", "monthly_charges": -5})
    print("đầu vào sai      →", bad.status_code, [(e["loc"][-1], e["msg"]) for e in bad.json()["detail"]])
    batch = client.post("/v1/predict/batch", json={"customers": records}).json()
    too_big = client.post("/v1/predict/batch", json={"customers": records * 6})
    print("lô 1200 dòng     →", too_big.status_code, too_big.json()["detail"][0]["msg"])

**Kiểm thử parity**: xác suất trả về qua API (sau khi đi qua JSON → pydantic → DataFrame) phải khớp đúng với mô hình gốc trên cùng dữ liệu. Đây là kiểm thử chống lệch train/serving quan trọng nhất (chương 39).

In [ ]:
api_probs = np.array([p["churn_probability"] for p in batch])
direct_probs = model.predict_proba(X_te.iloc[:200])[:, 1]
print(f"sai khác lớn nhất API vs mô hình gốc: {np.abs(api_probs - direct_probs).max():.2e} | khớp: {np.allclose(api_probs, direct_probs)}")
print("mọi dự đoán mang model_version:", {p["model_version"] for p in batch})

## 5. Chạy server thật

Trong production, dịch vụ chạy bằng một ASGI server — **uvicorn** — trong tiến trình riêng: `uvicorn app:app --host 0.0.0.0 --port 8000 --workers 4`. Ta khởi động nó như một tiến trình con, gọi qua HTTP thật bằng `httpx`, rồi tắt.

In [ ]:
def free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


port = free_port()
BASE = f"http://127.0.0.1:{port}"
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--port", str(port), "--log-level", "warning"],
                          cwd=SERVICE_DIR)
for _ in range(100):                                                   # chờ server sẵn sàng (readiness)
    try:
        if httpx.get(f"{BASE}/health", timeout=0.5).json()["model_loaded"]:
            break
    except httpx.TransportError:
        time.sleep(0.2)
print("server sẵn sàng tại", BASE, "| tài liệu tương tác:", f"{BASE}/docs")
print("các endpoint (từ OpenAPI):", list(httpx.get(f"{BASE}/openapi.json").json()["paths"]))

In [ ]:
def timed_calls(client, n, path, payload):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        client.post(path, json=payload).raise_for_status()
        times.append((time.perf_counter() - t0) * 1000)
    return np.array(times)


with httpx.Client(base_url=BASE) as http:
    timed_calls(http, 20, "/v1/predict", records[0])                   # khởi động (warm-up)
    single = timed_calls(http, 200, "/v1/predict", records[0])
    batched = timed_calls(http, 30, "/v1/predict/batch", {"customers": records[:100]})
print(f"đơn lẻ        : p50 = {np.percentile(single, 50):5.2f} ms | p99 = {np.percentile(single, 99):5.2f} ms | {1000 / single.mean():6.0f} dự đoán/giây")
print(f"lô 100 khách  : p50 = {np.percentile(batched, 50):5.2f} ms | p99 = {np.percentile(batched, 99):5.2f} ms | {100 * 1000 / batched.mean():6.0f} dự đoán/giây")

In [ ]:
import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [3, 2]})
axes[0].hist(single, bins=40, color="C0")
for q, ls in [(50, "-"), (99, "--")]:
    axes[0].axvline(np.percentile(single, q), color="C3", ls=ls, label=f"p{q} = {np.percentile(single, q):.2f} ms")
axes[0].set_xlabel("độ trễ một request đơn lẻ qua HTTP (ms)")
axes[0].set_title("Phân phối độ trễ: theo dõi p99, không chỉ trung bình")
axes[0].legend()
axes[1].bar(["đơn lẻ", "lô 100 khách"], [1000 / single.mean(), 100 * 1000 / batched.mean()], color=["C0", "C2"])
axes[1].set_ylabel("dự đoán / giây")
axes[1].set_title("Thông lượng")
plt.tight_layout()
plt.show()

- Độ trễ qua HTTP gồm: mạng, phân tích JSON, kiểm tra pydantic, tạo DataFrame, mô hình, ghi log, tuần tự hoá phản hồi — mô hình chỉ là **một phần**.
- Gom lô tăng **thông lượng** gấp nhiều lần (chi phí cố định mỗi request được chia cho 100 dòng) — nhưng tăng độ trễ của từng request. Batch scoring hằng đêm nên dùng lô lớn (hoặc chạy thẳng trên bảng dữ liệu, không qua API).
- Một tiến trình Python xử lý tác vụ CPU gần như tuần tự (GIL) → tăng thông lượng bằng **nhiều worker** (`--workers`), nhiều bản sao sau load balancer (chương 40 bài 02, 04).

**Anti-pattern: nạp mô hình trong mỗi request.**

In [ ]:
forest = make_pipeline(ColumnTransformer([("num", StandardScaler(), NUMERIC), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
                       RandomForestClassifier(300, random_state=0, n_jobs=1)).fit(X_tr, y_tr)
forest_path = SERVICE_DIR / "forest.joblib"
joblib.dump(forest, forest_path)
for name, m, path in [("hồi quy logistic", model, model_path), ("rừng 300 cây", forest, forest_path)]:
    t0 = time.perf_counter()
    for _ in range(5):
        joblib.load(path)
    load_ms = (time.perf_counter() - t0) / 5 * 1000
    t0 = time.perf_counter()
    for _ in range(20):
        m.predict_proba(X_te.iloc[:1])
    predict_ms = (time.perf_counter() - t0) / 20 * 1000
    print(f"{name:17s}: file {path.stat().st_size / 2**20:6.1f} MB | nạp {load_ms:7.1f} ms | dự đoán 1 dòng {predict_ms:5.1f} ms"
          f" → nạp mỗi request chậm ~{(load_ms + predict_ms) / predict_ms:.0f}×")
forest_path.unlink()

In [ ]:
server.terminate()                                                     # production: tắt êm (SIGTERM) để xử lý xong request đang chạy
server.wait(timeout=10)
print("đã tắt server")

## 6. Log dự đoán

Mỗi dự đoán đã được ghi một dòng JSON. Đây là **nguyên liệu** của giám sát (bài 03): phân phối đặc trưng và dự đoán theo thời gian, độ trễ, và — khi nhãn thật về (khách có rời bỏ không, sau 30 ngày) — nối theo `request_id` để tính hiệu năng thật.

In [ ]:
log = pd.read_json(SERVICE_DIR / "predictions.jsonl", lines=True)
features_logged = pd.json_normalize(log.features)
print(f"{len(log):,} dòng log | phiên bản mô hình: {log.model_version.unique()} | cột: {list(log.columns)}")
print(pd.concat([log[["request_id", "churn_probability", "latency_ms"]], features_logged[["contract", "monthly_charges"]]], axis=1).head(3).round(3).to_string(index=False))

Trong production, log đi vào hệ thống thu thập (Kafka, Kinesis, Pub/Sub → kho dữ liệu) thay vì file; cẩn trọng với **dữ liệu cá nhân** trong log (ẩn danh hoá, thời hạn lưu, quyền truy cập).

## 7. Checklist dịch vụ production

| Hạng mục | Thực hành |
|---|---|
| Hợp đồng | Phiên bản trong URL; schema chặt; trả `model_version`, `request_id` |
| Khởi động | Nạp mô hình một lần, kiểm tra hash/phiên bản; readiness ≠ liveness (`/ready` chỉ OK khi mô hình đã nạp) |
| Đầu vào | Giới hạn kích thước request/lô; timeout; 422 với mô tả rõ, không lộ stack trace |
| Hiệu năng | `def` cho tác vụ CPU; nhiều worker; gom lô khi có thể; đo p50/p99 qua HTTP |
| Bảo mật | Xác thực (API key/OAuth), HTTPS, giới hạn tần suất; không log dữ liệu nhạy cảm thô |
| Quan sát | Log có cấu trúc; metric (số request, lỗi, độ trễ — bài 03); truy vết theo request_id |
| Vận hành | Tắt êm (graceful shutdown); cấu hình qua biến môi trường (`MODEL_DIR`); đóng gói Docker (bài 02) |

Lựa chọn khác FastAPI: BentoML, Ray Serve, TorchServe, NVIDIA Triton (GPU, nhiều framework), KServe/Seldon (Kubernetes), dịch vụ quản lý (SageMaker, Vertex AI) — cùng các khái niệm trên.

## 🧠 Kiểm tra nhanh

1. Vì sao API nên trả `model_version` và `request_id`?
2. Vì sao nạp mô hình trong `lifespan` thay vì trong hàm xử lý request?
3. Khi nào dùng `def`, khi nào `async def` cho endpoint?
4. Kiểm thử parity kiểm tra điều gì? Vì sao quan trọng?
5. Gom lô ảnh hưởng thế nào tới độ trễ và thông lượng?
6. Log dự đoán dùng để làm gì?

## 📌 Tóm tắt

- Hợp đồng API trước, code sau: endpoint có phiên bản, schema chặt, mã lỗi rõ.
- Dịch vụ: nạp mô hình một lần (kiểm tra hash), pydantic kiểm tra đầu vào, endpoint `def` cho tác vụ CPU, ghi log mỗi dự đoán.
- Kiểm thử bằng TestClient (nhanh, chạy trong CI) gồm kiểm thử parity; đo độ trễ qua HTTP thật.
- Lô tăng thông lượng; nhiều worker để mở rộng; log dự đoán là nền của giám sát.

## 📚 Tài liệu tham khảo

- Tài liệu FastAPI — fastapi.tiangolo.com (Lifespan events, Testing, Concurrency and async/await)
- Tài liệu Pydantic v2 — docs.pydantic.dev
- Huyen (2022) — *Designing Machine Learning Systems*, chương 7 (Model Deployment and Prediction Service)
- Google Cloud — *MLOps: Continuous delivery and automation pipelines in machine learning*